In [ ]:
# !pip install torch_geometric

In [ ]:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

In [ ]:
# import all libraries needed downstream
import os
import gc
import wandb
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
import scipy
from sklearn.metrics import mean_squared_error
import math
import networkx as nx
import seaborn as sns
import time
from torch.nn import Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.nn import ChebConv, GraphConv, GCNConv, TAGConv, GATConv
from torch_geometric.data import Data
from torch.utils.data import TensorDataset
from torch_geometric.loader import DataLoader
import torch_scatter
from typing import Dict, Tuple, List, Optional
import matplotlib.pyplot as plt
from torch_scatter import scatter_softmax, scatter_sum
import scipy.sparse as sp
import scipy.sparse.linalg as spla

In [ ]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
!nvidia-smi

In [ ]:
system_size = 118

In [ ]:
# Load the data
data = np.load(f'/home/oarowolo/workfile/OPFData/{system_size}bus_nminusone_combined_dataset.npz',allow_pickle=True)

# Get all keys
print("Available keys in the dataset:")
for key in data.files:
    # Print the key and its array shape
    print(f"{key}: shape {data[key].shape}")

In [ ]:
def compute_gandb(edge_iputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
def get_B_matrix(N, edges, edge_weights):
    # Create a zero tensor of shape (N,N)
    B_matrix = np.zeros((N, N))
    
    # Unpack the edges into source and destination nodes
    sources, destinations = zip(*edges)
    
    # Use advanced indexing to place weights in the right spots
    B_matrix[sources, destinations] = edge_weights.squeeze()
    B_matrix[destinations, sources] = edge_weights.squeeze()
    return B_matrix

In [ ]:
def adjacency_to_laplacian(B_adj):

    # Ensure matrix is square
    assert B_adj.shape[0] == B_adj.shape[1], "Input must be square"

    # Copy to avoid modifying original
    B_laplacian = B_adj.copy()

    # Set diagonal as row sum of adjacency (i.e., degree)
    np.fill_diagonal(B_laplacian, -B_adj.sum(axis=1))

    return B_laplacian


In [ ]:
def compute_row_stats_from_laplacian(L, tol=1e-8):
    """
    More efficient approach: precompute pseudoinverse diagonal, then process rows.
    """
    L = sp.csc_matrix(L)
    N = L.shape[0]
    
    # Remove reference node and compute reduced inverse
    keep = np.arange(N - 1)
    L_reduced = L[np.ix_(keep, keep)]
    L_reduced_inv = spla.inv(L_reduced.tocsc()).toarray()
    
    # Expand to full pseudoinverse
    L_plus = np.zeros((N, N))
    L_plus[np.ix_(keep, keep)] = L_reduced_inv
    
    # Apply projection
    I = np.eye(N)
    ones = np.ones((N, N)) / N
    L_plus = (I - ones) @ L_plus @ (I - ones)
    
    # Compute effective resistance matrix and find global max
    diag = np.diag(L_plus)
    R = diag[:, None] + diag[None, :] - 2 * L_plus
    
    # Find global maximum (excluding diagonal which should be ~0)
    mask = ~np.eye(N, dtype=bool)
    global_max = np.max(R[mask])
    
    # Normalize and compute statistics
    R_normalized = R / global_max
    stats_matrix = np.zeros((N, 5))
    
    for i in range(N):
        row_no_diag = R_normalized[i, mask[i]]
        stats_matrix[i, 0] = np.mean(row_no_diag)
        stats_matrix[i, 1] = np.median(row_no_diag)
        stats_matrix[i, 2] = np.std(row_no_diag)
        stats_matrix[i, 3] = np.max(row_no_diag)
        stats_matrix[i, 4] = np.min(row_no_diag)
    
    return stats_matrix

In [ ]:
## wandb set-up
api_key = '' ###insert your own api key
wandb.login(key=api_key)

In [ ]:
run = wandb.init(
      # Set the project where this run will be logged
      project="Towards_Generalization_of_GNN_for_ACOPF",
      # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
      name=f"{system_size}_N-1_HybridHeteroGNN_5_256_PQVT",
      # Track hyperparameters and run metadata
      config={
      "architecture": "GNN",
      "dataset": "N-1",
      "epochs": 100,
      })

In [ ]:
grid_bus = data['grid_bus']  
grid_generator = data['grid_generator']
grid_load = data['grid_load']
grid_shunt = data['grid_shunt']
grid_ac_line_features = data['grid_ac_line_features']
grid_transformer_features = data['grid_transformer_features']
grid_ac_line_receivers = data['grid_ac_line_receivers']
grid_ac_line_senders = data['grid_ac_line_senders']
grid_transformer_senders = data['grid_transformer_senders']
grid_transformer_receivers = data['grid_transformer_receivers']
grid_generator_link_senders = data['grid_generator_link_senders']
grid_generator_link_receivers = data['grid_generator_link_receivers']
solution_bus = data['solution_bus']  
solution_generator = data['solution_generator'] 
solution_objective = data['metadata_objective']
solution_objective = solution_objective.reshape(-1,1)
grid_load_link_receivers = data['grid_load_link_receivers']
grid_load_link_senders = data['grid_load_link_senders']
grid_shunt_link_receivers = data['grid_shunt_link_receivers']
grid_shunt_link_senders = data['grid_shunt_link_senders']

In [ ]:
grid_bus_list = []
grid_generator_list = []
grid_load_list = []
grid_shunt_list = []
grid_ac_line_features_list  = []
grid_transformer_features_list = []
solution_bus_list = []
solution_generator_list = []
grid_ac_line_senders_list = []
grid_ac_line_receivers_list = []
grid_transformer_senders_list = []
grid_transformer_receivers_list = []
load_indices_list = []
shunt_indices_list = []
generator_indices_list = []
master_branch_list = []
pe_list = []
# pe_list = torch.load(f'bus_{system_size}_normed_pe_encodings.pt')  we can load pe_list instead if it has been precomputed

In [ ]:
# for k in range(grid_bus.shape[0]):
for k in tqdm(range(grid_bus.shape[0]), desc="Data Processing Progress"):   

    generator_indices = grid_generator_link_receivers[k].astype(int)
    generator_indices_list.append(generator_indices)
    load_indices = grid_load_link_receivers[k].astype(int)
    load_indices_list.append(load_indices)
    shunt_indices = grid_shunt_link_receivers[k].astype(int)
    shunt_indices_list.append(shunt_indices)
    
    grid_bus_k = grid_bus[k].astype(np.float32)
    grid_bus_list.append(grid_bus_k)
    grid_generator_k = grid_generator[k].astype(np.float32)
    grid_generator_list.append(grid_generator_k)
    grid_load_k = grid_load[k].astype(np.float32)
    grid_load_list.append(grid_load_k)
    grid_shunt_k = grid_shunt[k].astype(np.float32)
    grid_shunt_list.append(grid_shunt_k)
    
    solution_bus_k = solution_bus[k].astype(np.float32)
    solution_bus_list.append(solution_bus_k)
    solution_generator_k = solution_generator[k].astype(np.float32)
    solution_generator_list.append(solution_generator_k)

    grid_transformer_sender_k = grid_transformer_senders[k].astype(np.float32)
    grid_transformer_senders_list.append(grid_transformer_sender_k)
    grid_transformer_receiver_k = grid_transformer_receivers[k].astype(np.float32)
    grid_transformer_receivers_list.append(grid_transformer_receiver_k)
    grid_ac_line_sender_k = grid_ac_line_senders[k].astype(np.float32)
    grid_ac_line_senders_list.append(grid_ac_line_sender_k)
    grid_ac_line_receiver_k = grid_ac_line_receivers[k].astype(np.float32)
    grid_ac_line_receivers_list.append(grid_ac_line_receiver_k)

    grid_transformer_features_k = grid_transformer_features[k].astype(np.float32)
    grid_transformer_features_list.append(grid_transformer_features_k)
    grid_ac_line_features_k = grid_ac_line_features[k].astype(np.float32)
    grid_ac_line_features_list.append(grid_ac_line_features_k)
    
    branch_list = list(zip(grid_ac_line_senders[k], grid_ac_line_receivers[k]))
    transformer_list = list(zip(grid_transformer_senders[k], grid_transformer_receivers[k]))
    for j in transformer_list:
        branch_list.append(j)
    master_branch_list.append(branch_list)

    ################################ this is where we create the positional encoding stuff
    edge_inputs = np.zeros((len(branch_list),11))
    edge_inputs[:grid_ac_line_features_k.shape[0],:9] = grid_ac_line_features_k  # rearranging edge inputs to align for transformers and transmission lines
    edge_inputs[grid_ac_line_features_k.shape[0]:,:2] =  grid_transformer_features_k[:,:2]
    edge_inputs[grid_ac_line_features_k.shape[0]:,2:4] =  grid_transformer_features_k[:,9:]
    edge_inputs[grid_ac_line_features_k.shape[0]:,4:9] =  grid_transformer_features_k[:,2:7]
    edge_inputs[grid_ac_line_features_k.shape[0]:,9:] =  grid_transformer_features_k[:,7:9]
    edge_inputs[:grid_ac_line_features_k.shape[0],9:10] = 1.0

    edge_g, edge_b = compute_gandb(edge_inputs)
    B_weighted = get_B_matrix(system_size, branch_list,edge_b)
    b_mat = B_weighted
    B_lap = adjacency_to_laplacian(b_mat)
    raw_PE = compute_row_stats_from_laplacian(B_lap)
    bus_pe = torch.tensor(raw_PE,dtype=torch.float)
    pe_list.append([bus_pe])

In [ ]:
# flatter_pe_list = [j[0] for j in pe_list]
# torch.save(flatter_pe_list, f'bus_{system_size}_pe_encodings.pt')

In [ ]:
gen_nums = [arr.shape for arr in grid_generator_link_receivers] 
gen_max = max(gen_nums)

In [ ]:
missing_gen_indices = [i for i, arr in enumerate(grid_generator_link_receivers) if arr.shape[0] == gen_max[0]-1]

In [ ]:
len(missing_gen_indices)

In [ ]:
len(grid_generator_link_receivers)

In [ ]:
solution_objective.shape

In [ ]:
solution_objective = np.array(solution_objective.tolist(), dtype=np.float32)

In [ ]:
solution_objective.mean()

In [ ]:
batch_size = 16

In [ ]:
from torch_geometric.data import HeteroData

def create_grid_hetero_data(
    grid_bus,                    
    grid_generator,              
    grid_load,                   
    grid_shunt,                  
    grid_ac_line_features,       
    grid_transformer_features,   
    grid_ac_line_senders,        
    grid_ac_line_receivers,      
    grid_transformer_senders,    
    grid_transformer_receivers,  
    generator_indices,          
    load_indices,                
    shunt_indices,               
    solution_bus,                
    solution_generator,
    pe,
    batch_idx=0
):
    
    # Create a new HeteroData instance
    data = HeteroData()
    
    # Process a single batch if specified, otherwise we'd need to handle batching differently
    if batch_idx is not None:
        # Extract features for the specified batch
        bus_features = torch.tensor(grid_bus[batch_idx], dtype=torch.float)
        generator_features = torch.tensor(grid_generator[batch_idx], dtype=torch.float)
        load_features = torch.tensor(grid_load[batch_idx], dtype=torch.float)
        shunt_features = torch.tensor(grid_shunt[batch_idx], dtype=torch.float)


        bus_pe = pe[batch_idx].to(torch.float)
        
        ac_line_features = torch.tensor(grid_ac_line_features[batch_idx], dtype=torch.float)
        transformer_features = torch.tensor(grid_transformer_features[batch_idx], dtype=torch.float)
        
        # Extract solution values for the specified batch
        bus_solutions = torch.tensor(solution_bus[batch_idx], dtype=torch.float)
        generator_solutions = torch.tensor(solution_generator[batch_idx], dtype=torch.float)
        
        # Add node features
        data['bus'].x = bus_features
        data['generator'].x = generator_features
        data['load'].x = load_features
        data['shunt'].x = shunt_features

        #create global IDs for nodes to make indexing of transformer easier
        data['bus'].graph_id = torch.full((bus_features.shape[0],), batch_idx, dtype=torch.long)
        data['generator'].graph_id = torch.full((generator_features.shape[0],), batch_idx, dtype=torch.long)
        data['load'].graph_id = torch.full((load_features.shape[0],), batch_idx, dtype=torch.long)
        data['shunt'].graph_id = torch.full((shunt_features.shape[0],), batch_idx, dtype=torch.long)

        ####use indices specially
        g_indices = generator_indices[batch_idx]
        s_indices = shunt_indices[batch_idx]
        l_indices = load_indices[batch_idx]

        #Add node positional encoding
        data['bus'].pe = bus_pe
        data['generator'].pe = bus_pe[g_indices]  ## added arbitrary values to distinguish buses from special nodes
        data['load'].pe = bus_pe[l_indices]
        data['shunt'].pe = bus_pe[s_indices]
        
        # Add solution values as target values (y)
        data['bus'].y = bus_solutions
        data['generator'].y = generator_solutions
        
        # Add edge indices and features for AC lines (bus to bus)
        senders = torch.tensor(grid_ac_line_senders[batch_idx].flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_ac_line_receivers[batch_idx].flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'ac_line', 'bus'].edge_index = edge_index
        data['bus', 'ac_line', 'bus'].edge_attr = ac_line_features
        
        # Add edge indices and features for transformers (bus to bus)
        senders = torch.tensor(grid_transformer_senders[batch_idx].flatten(), dtype=torch.long)
        receivers = torch.tensor(grid_transformer_receivers[batch_idx].flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'transformer', 'bus'].edge_index = edge_index
        data['bus', 'transformer', 'bus'].edge_attr = transformer_features
        
        # Add pseudo-edges from generators to buses
        gen_to_bus = torch.tensor(generator_indices[batch_idx].flatten(), dtype=torch.long)
        gen_indices = torch.arange(len(gen_to_bus), dtype=torch.long)
        gen_edge_index = torch.stack([gen_indices, gen_to_bus], dim=0)
        data['generator', 'connects_to', 'bus'].edge_index = gen_edge_index
        data['generator', 'connects_to', 'bus'].edge_attr = torch.ones((len(gen_to_bus), 3))
        
        # Add pseudo-edges from loads to buses
        load_to_bus = torch.tensor(load_indices[batch_idx].flatten(), dtype=torch.long)
        load_index = torch.arange(len(load_to_bus), dtype=torch.long)
        load_edge_index = torch.stack([load_index, load_to_bus], dim=0)
        data['load', 'connects_to', 'bus'].edge_index = load_edge_index
        data['load', 'connects_to', 'bus'].edge_attr = torch.ones((len(load_to_bus), 3))
        
        # Add pseudo-edges from shunts to buses
        shunt_to_bus = torch.tensor(shunt_indices[batch_idx].flatten(), dtype=torch.long)
        shunt_index = torch.arange(len(shunt_to_bus), dtype=torch.long)
        shunt_edge_index = torch.stack([shunt_index, shunt_to_bus], dim=0)
        data['shunt', 'connects_to', 'bus'].edge_index = shunt_edge_index
        data['shunt', 'connects_to', 'bus'].edge_attr = torch.ones((len(shunt_to_bus), 3))
    
    else:
        # Handle all batches (would require batching approach)
        raise NotImplementedError("Processing all batches at once is not implemented in this example")
    
    return data


def create_dataloader(
    grid_bus,
    grid_generator,
    grid_load,
    grid_shunt,
    grid_ac_line_features,
    grid_transformer_features,
    grid_ac_line_senders,
    grid_ac_line_receivers,
    grid_transformer_senders,
    grid_transformer_receivers,
    generator_indices,
    load_indices,
    shunt_indices,
    solution_bus,
    solution_generator,
    pe,
    batch_size=batch_size,
    shuffle = True
):
    
    # Create a list of HeteroData objects
    dataset = []
    
    for i in range(len(grid_bus)):  # Process up to 1000 samples for this example
        data = create_grid_hetero_data(
            grid_bus, 
            grid_generator,
            grid_load,
            grid_shunt,
            grid_ac_line_features,
            grid_transformer_features,
            grid_ac_line_senders,
            grid_ac_line_receivers,
            grid_transformer_senders,
            grid_transformer_receivers,
            generator_indices,
            load_indices,
            shunt_indices,
            solution_bus,
            solution_generator,
            pe,
            batch_idx=i
        )
        dataset.append(data)
    
    # Create a DataLoader
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,num_workers=min(8, torch.get_num_threads()))
    
    return loader

In [ ]:
def calculate_angle_differences(angles, edges):
  
    # Ensure angles are a NumPy array
    angles = np.asarray(angles)
    
    # Initialize array to store angle differences
    angle_differences = np.zeros(len(edges), dtype=np.float32)
    
    # Calculate angle differences for each edge
    for i, (node1, node2) in enumerate(edges):
        angle_differences[i] = angles[node2] - angles[node1]
    
    return angle_differences

In [ ]:
def train_val_test_split(train_ratio=0.9, val_ratio=0.05, test_ratio=0.05, seed=42):

    # Set random seed for reproducibility
    torch.manual_seed(seed)

    # Shuffle indices
    num_samples = 300000
    indices = torch.randperm(num_samples)

    # Compute split sizes
    train_size = int(train_ratio * num_samples)
    val_size = int(val_ratio * num_samples)

    # Split indices
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    return train_indices, val_indices, test_indices

In [ ]:
train_indices, val_indices, test_indices = train_val_test_split()

In [ ]:
train_grid_bus =  list(grid_bus_list[i] for i in train_indices)
train_grid_generator=  list(grid_generator_list[i] for i in train_indices)
train_grid_load=  list(grid_load_list[i] for i in train_indices)
train_grid_shunt=  list(grid_shunt_list[i] for i in train_indices)
train_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in train_indices)
train_grid_transformer_features=  list(grid_transformer_features_list[i] for i in train_indices)
train_grid_ac_line_senders =  list(grid_ac_line_senders_list[i] for i in train_indices)
train_grid_ac_line_receivers=  list(grid_ac_line_receivers_list[i] for i in train_indices)
train_grid_transformer_senders=  list(grid_transformer_senders_list[i] for i in train_indices)
train_grid_transformer_receivers=  list(grid_transformer_receivers_list[i] for i in train_indices)
train_generator_indices=  list(generator_indices_list[i] for i in train_indices)
train_load_indices=  list(load_indices_list[i] for i in train_indices)
train_shunt_indices=  list(shunt_indices_list[i] for i in train_indices)
train_solution_bus=  list(solution_bus_list[i] for i in train_indices)
train_solution_generator=  list(solution_generator_list[i] for i in train_indices)
train_bus_pe = [pe_list[i] for i in train_indices]

In [ ]:
train_loader= create_dataloader(train_grid_bus,
                                train_grid_generator,
                                train_grid_load,
                                train_grid_shunt,
                                train_grid_ac_line_features,
                                train_grid_transformer_features,
                                train_grid_ac_line_senders,
                                train_grid_ac_line_receivers,
                                train_grid_transformer_senders,
                                train_grid_transformer_receivers,
                                train_generator_indices,
                                train_load_indices,
                                train_shunt_indices,
                                train_solution_bus,
                                train_solution_generator,
                                train_bus_pe,
                                batch_size=batch_size,
                                shuffle = True)

In [ ]:
del train_grid_bus  
del train_grid_generator 
del train_grid_load 
del train_grid_shunt 
del train_grid_ac_line_features 
del train_grid_transformer_features 
del train_grid_ac_line_senders 


In [ ]:
del train_grid_ac_line_receivers 
del train_grid_transformer_senders 
del train_grid_transformer_receivers 
del train_generator_indices
del train_load_indices
del train_shunt_indices
del train_solution_bus
del train_solution_generator
del train_bus_pe
gc.collect()

In [ ]:
validate_grid_bus =  list(grid_bus_list[i] for i in val_indices)
validate_grid_generator=  list(grid_generator_list[i] for i in val_indices)
validate_grid_load=  list(grid_load_list[i] for i in val_indices)
validate_grid_shunt=  list(grid_shunt_list[i] for i in val_indices)
validate_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in val_indices)
validate_grid_transformer_features=  list(grid_transformer_features_list[i] for i in val_indices)
validate_grid_ac_line_senders=  list(grid_ac_line_senders_list[i] for i in val_indices)
validate_grid_ac_line_receivers=  list(grid_ac_line_receivers_list[i] for i in val_indices)
validate_grid_transformer_senders=  list(grid_transformer_senders_list[i] for i in val_indices)
validate_grid_transformer_receivers=  list(grid_transformer_receivers_list[i] for i in val_indices)
validate_generator_indices=  list(generator_indices_list[i] for i in val_indices)
validate_load_indices=  list(load_indices_list[i] for i in val_indices)
validate_shunt_indices=  list(shunt_indices_list[i] for i in val_indices)
validate_solution_bus=  list(solution_bus_list[i] for i in val_indices)
validate_solution_generator=  list(solution_generator_list[i] for i in val_indices)
validate_bus_pe = list(pe_list[i] for i in val_indices)

In [ ]:
val_loader =  create_dataloader(validate_grid_bus,
                                validate_grid_generator,
                                validate_grid_load,
                                validate_grid_shunt,
                                validate_grid_ac_line_features,
                                validate_grid_transformer_features,
                                validate_grid_ac_line_senders,
                                validate_grid_ac_line_receivers,
                                validate_grid_transformer_senders,
                                validate_grid_transformer_receivers,
                                validate_generator_indices,
                                validate_load_indices,
                                validate_shunt_indices,
                                validate_solution_bus,
                                validate_solution_generator,
                                validate_bus_pe,
                                batch_size=batch_size,
                                shuffle = False)

In [ ]:
del validate_grid_bus  
del validate_grid_generator 
del validate_grid_load 
del validate_grid_shunt 
del validate_grid_ac_line_features 
del validate_grid_transformer_features 
del validate_grid_ac_line_senders 
del validate_grid_ac_line_receivers 
del validate_grid_transformer_senders 
del validate_grid_transformer_receivers 
del validate_generator_indices
del validate_load_indices
del validate_shunt_indices
del validate_solution_bus
del validate_solution_generator
del validate_bus_pe
gc.collect()

In [ ]:
test_grid_bus =  list(grid_bus_list[i] for i in test_indices)
test_grid_generator=  list(grid_generator_list[i] for i in test_indices)
test_grid_load=  list(grid_load_list[i] for i in test_indices)
test_grid_shunt=  list(grid_shunt_list[i] for i in test_indices)
test_grid_ac_line_features=  list(grid_ac_line_features_list[i] for i in test_indices)
test_grid_transformer_features=  list(grid_transformer_features_list[i] for i in test_indices)
test_grid_ac_line_senders=  list(grid_ac_line_senders_list[i] for i in test_indices)
test_grid_ac_line_receivers=  list(grid_ac_line_receivers_list[i] for i in test_indices)
test_grid_transformer_senders=  list(grid_transformer_senders_list[i] for i in test_indices)
test_grid_transformer_receivers=  list(grid_transformer_receivers_list[i] for i in test_indices)
test_generator_indices=  list(generator_indices_list[i] for i in test_indices)
test_load_indices=  list(load_indices_list[i] for i in test_indices)
test_shunt_indices=  list(shunt_indices_list[i] for i in test_indices)
test_solution_bus=  list(solution_bus_list[i] for i in test_indices)
test_solution_generator=  list(solution_generator_list[i] for i in test_indices)
test_bus_pe = list(pe_list[i] for i in test_indices)

In [ ]:
test_loader = create_dataloader(test_grid_bus,
                                test_grid_generator,
                                test_grid_load,
                                test_grid_shunt,
                                test_grid_ac_line_features,
                                test_grid_transformer_features,
                                test_grid_ac_line_senders,
                                test_grid_ac_line_receivers,
                                test_grid_transformer_senders,
                                test_grid_transformer_receivers,
                                test_generator_indices,
                                test_load_indices,
                                test_shunt_indices,
                                test_solution_bus,
                                test_solution_generator,
                                test_bus_pe,
                                batch_size=batch_size,
                                shuffle = False)

In [ ]:
# del grid_bus  
# del grid_generator 
# del grid_load 
# del grid_shunt 
# del grid_ac_line_features 
# del grid_transformer_features 
# del grid_ac_line_receivers 
# del grid_ac_line_senders 
# del grid_transformer_senders 
# del grid_transformer_receivers 
# del grid_generator_link_senders 
# del grid_generator_link_receivers 
# del solution_bus  
# del solution_generator 
# del grid_load_link_receivers 
# del grid_load_link_senders 
# del grid_shunt_link_receivers
# del grid_shunt_link_senders
# del pe_list
# gc.collect()

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size, layers, layernorm=True, use_leaky=False): 
        super().__init__()
        # Use Sequential instead of ModuleList for faster forward pass
        modules = []
        for i in range(layers):
            modules.append(torch.nn.Linear(
                input_size if i == 0 else hidden_size,
                output_size if i == layers - 1 else hidden_size,
            ))
            if i != layers - 1:
                modules.append(torch.nn.ReLU())
            if use_leaky:
                modules.append(torch.nn.LeakyReLU(negative_slope=0.02))
        if layernorm:
            modules.append(torch.nn.LayerNorm(output_size))
        
        self.network = torch.nn.Sequential(*modules)

    def forward(self, x):
        # Sequential is faster than iterating through ModuleList
        return self.network(x)

In [ ]:
# from performer_pytorch import SelfAttention
from torch_geometric.utils import to_dense_batch
from torch_geometric.nn.attention import PerformerAttention

class HeteroPerformerLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.attn = PerformerAttention(
            channels=hidden_dim,
            heads=num_heads
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Post-attention MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
    def forward(self, x_dict, xm_dict, batch_dict):
            # Flatten all node types
            flat_x, flat_xm, flat_batch = [], [], []
            slices = {}
            offset = 0
    
            for ntype in x_dict:
                x = x_dict[ntype]
                xm = xm_dict[ntype]
                b = batch_dict[ntype]  # batch indices for each node
    
                slices[ntype] = slice(offset, offset + x.size(0))
                flat_x.append(x)
                flat_xm.append(xm)
                flat_batch.append(b)
                offset += x.size(0)
    
            x_all = torch.cat(flat_x, dim=0)         # [N, D]
            xm_all = torch.cat(flat_xm, dim=0)       # [N, D]
            global_batch_all = torch.cat(flat_batch, dim=0) # [N]

            _, batch_all = torch.unique(global_batch_all, return_inverse=True)
            sorted_indices = torch.argsort(batch_all) # resort batch index to be ascending but that means I have to sort the xs myself too
            batch_all_sorted = batch_all[sorted_indices]
            x_all_sorted = x_all[sorted_indices]
            xm_all_sorted = xm_all[sorted_indices]
            # Convert to [B, N_max, D] and mask            
            x_dense, mask = to_dense_batch(x_all_sorted, batch_all_sorted)   # [B, N, D], [B, N]
            
            # Apply masked Performer attention
            x_attn = self.attn(x_dense, mask=mask)              # [B, N, D]
            # Residual + Norm
            xt_out = self.norm1(x_dense + x_attn)          
            # Unpad: [real_nodes, D]
            xt_out = xt_out[mask]
    
            x_comb = self.norm2(xt_out + xm_all_sorted)
            x_final = self.mlp(x_comb)

            # dont forget to give x_final it unsorted arrangement
            unsorted_x = torch.empty_like(x_final)
            unsorted_x[sorted_indices] = x_final
            x_final = unsorted_x
            # x_final = x_final[sorted_indices]

            # Unflatten by slice
            return {ntype: x_final[slices[ntype]] for ntype in x_dict.keys()}


In [ ]:
class HeteroInteractionNetwork(nn.Module):
    def __init__(self, node_types, edge_types,physical_edge_types, hidden_size, layers):
        super().__init__()
        
        self.physical_edge_types = physical_edge_types
        self.edge_updaters = nn.ModuleDict()
        for src, rel, dst in edge_types:
            edge_key = f"{src}_{rel}_{dst}"
            edge_type = (src, rel, dst)
            
            # For physical edges (ac_line and transformer), use node + edge features
            if edge_type in physical_edge_types:
                self.edge_updaters[edge_key] = MLP(hidden_size * 3, hidden_size, hidden_size, layers)
            else:
                # For other edge types, only use node features
                self.edge_updaters[edge_key] = MLP(hidden_size * 2, hidden_size, hidden_size, layers)
        
        # Create a node updater for each node type
        self.node_updaters = nn.ModuleDict({
            node_type: MLP(hidden_size * 2, hidden_size, hidden_size, layers)
            for node_type in node_types
        })

    
    def forward(self, x_dict, edge_indices_dict, edge_features_dict):
        # Store updated node and edge features
        updated_edge_features = {}
        
        # Prepare aggregated messages storage
        aggregated_messages = {node_type: torch.zeros_like(feat) 
                              for node_type, feat in x_dict.items()}
        
        # Process each edge type in parallel
        for edge_type, edge_index in edge_indices_dict.items():
            src_type, rel_type, dst_type = edge_type
            edge_key = f"{src_type}_{rel_type}_{dst_type}"
            
            # Get node features for this edge
            src, dst = edge_index
            x_i = x_dict[dst_type][dst]  # Destination nodes
            x_j = x_dict[src_type][src]  # Source nodes
            # Update edge features based on edge type
            if edge_type in self.physical_edge_types:
                # For physical edges, include edge features in message
                edge_feature = edge_features_dict[edge_type]
                edge_msg = torch.cat((x_i, x_j, edge_feature), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg)
                updated_edge_features[edge_type] =  updated_edge + edge_feature  
            else:
                # For non-physical edges, only use node features
                edge_msg = torch.cat((x_i, x_j), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg)
                
                #Check if we have existing edge features from previous layers
                # if edge_type in edge_features_dict:
                #     edge_feature = edge_features_dict[edge_type]
                #     updated_edge_features[edge_type] = edge_feature + updated_edge  ### removed residual connection for non-physical edges here
                # else:
                    # First layer - initialize with the computed edge features
                updated_edge_features[edge_type] = updated_edge 
            
            # Efficient message aggregation using torch_scatter
            aggregated_messages[dst_type] = torch_scatter.scatter_add(updated_edge, dst, dim=0, out=aggregated_messages[dst_type])
            aggregated_messages[src_type] = torch_scatter.scatter_add(updated_edge, src, dim=0, out=aggregated_messages[src_type])
        
        # Update node features
        updated_nodes = {}
        for node_type, x in x_dict.items():
            # Combine node features with aggregated messages
            node_input = torch.cat((x, aggregated_messages[node_type]), dim=-1)
            node_update = self.node_updaters[node_type](node_input)
            updated_nodes[node_type] = x + node_update 
        
        return updated_nodes, updated_edge_features


In [ ]:
class HeteroInteractGNN(torch.nn.Module):
    def __init__(
        self,
        hidden_size=256,
        n_mp_layers=5,
        bus_features=4,
        gen_features=11,
        load_features=2,
        shunt_features=2,
        ac_line_features=9,
        transformer_features=11,
        connects_to_features=3,
        output_dim=2
    ):
        super().__init__()
        
        # Define node and edge types
        self.node_types = ['bus', 'generator', 'load', 'shunt']
        self.edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus'),
            ('generator', 'connects_to', 'bus'),
            ('load', 'connects_to', 'bus'),
            ('shunt', 'connects_to', 'bus')
        ]

        self.physical_edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus')
        ]        
        #Node encoders - separate MLP for each node type
        self.node_encoders = nn.ModuleDict({
            'bus': MLP(bus_features, hidden_size, hidden_size-5, 2),
            'generator': MLP(gen_features, hidden_size, hidden_size-5, 2),
            'load': MLP(load_features, hidden_size, hidden_size-5, 2),
            'shunt': MLP(shunt_features, hidden_size, hidden_size-5, 2)
        })


        self.global_attn_layers = nn.ModuleList([
            HeteroPerformerLayer(hidden_size)
            for _ in range(n_mp_layers)
        ])

        #Edge encoders - separate MLP for each edge type
        self.edge_encoders = nn.ModuleDict({
            'ac_line': MLP(ac_line_features, hidden_size, hidden_size, 2),
            'transformer': MLP(transformer_features, hidden_size, hidden_size, 2)
        })
        
        # Interaction network layers
        self.n_mp_layers = n_mp_layers
        self.layers = torch.nn.ModuleList([
            HeteroInteractionNetwork(self.node_types, self.edge_types,self.physical_edge_types, hidden_size, 2)
            for _ in range(n_mp_layers)
        ])

        
        # Node decoders - separate for bus and generator
        self.node_decoders = nn.ModuleDict({
            'bus': MLP(hidden_size, hidden_size, output_dim, 2, layernorm=False),
            'generator': MLP(hidden_size, hidden_size, output_dim, 2,layernorm=False)
        })

    def forward(self, data):
        # Encode node features
        x_dict_init = {}
        x_dict = {}
        
        # Batch node encoding
        for node_type in self.node_types:
            if hasattr(data[node_type], 'x'):
                x_dict_init[node_type] = self.node_encoders[node_type](data[node_type].x)
                x_dict[node_type] = torch.cat([x_dict_init[node_type], data[node_type].pe], dim=-1)

        # Encode edge features
        edge_feature_dict = {}
        for src, rel, dst in self.edge_types:
            edge_type = (src, rel, dst)
            if (edge_type in self.physical_edge_types and 
                edge_type in data.edge_types and 
                hasattr(data[edge_type], 'edge_attr')):
                edge_feature_dict[edge_type] = self.edge_encoders[rel](data[edge_type].edge_attr)
        
        # Extract edge indices
        edge_index_dict = {
            edge_type: data[edge_type].edge_index
            for edge_type in self.edge_types
            if edge_type in data.edge_types and hasattr(data[edge_type], 'edge_index')
        }

        batch_dict = {
            ntype: data[ntype].graph_id
            for ntype in x_dict
        }
        

        # Apply message passing layers
        for i in range(self.n_mp_layers):
            xm_dict, edge_feature_dict = self.layers[i](x_dict, edge_index_dict, edge_feature_dict)
            x_dict = self.global_attn_layers[i](x_dict, xm_dict, batch_dict)

        # Apply decoders
        output = {}
        output['bus'] = torch.sigmoid(self.node_decoders['bus'](x_dict['bus']))
        output['generator'] = torch.sigmoid(self.node_decoders['generator'](x_dict['generator']))
        
        return output

In [ ]:
def convert_voltage_bounds(model_input):

    num_nodes = model_input.shape[0]

    vmin = model_input[:,2:3]

    vmax = model_input[:,3:4]

    thetamin =  torch.tensor([-2.00]).to(device)
    thetamin = thetamin.tile((num_nodes,1))

    thetamax =  torch.tensor([2.00]).to(device)
    thetamax = thetamax.tile((num_nodes,1))

    bounds_up = torch.concat((thetamax, vmax), dim=1)
    bounds_down = torch.concat((thetamin, vmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
def convert_power_bounds(model_input):

    num_nodes = model_input.shape[0]

    pmin = model_input[:,2:3]

    pmax = model_input[:,3:4]

    
    qmin = model_input[:,5:6]

    qmax = model_input[:,6:7]

    bounds_up = torch.concat((pmax, qmax), dim=1)
    bounds_down = torch.concat((pmin, qmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
# Example training step
def train_model(model, trainloader, optimizer):
    
    model.train()
    total_loss = 0
    criterion = nn.MSELoss()
    
    for batch in trainloader:
    
        optimizer.zero_grad()
        
        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)
        

        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)
        loss = criterion(combined_targets, combined_outputs)

        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(trainloader)

In [ ]:
# Example training step
def validate_model(model, val_loader):
    
    model.eval()
    total_loss = 0
    criterion = nn.MSELoss()
    
    for batch in val_loader:
        
        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)

        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)

        loss = criterion(combined_targets, combined_outputs)
     
        total_loss += loss.item()
        
    return total_loss / len(val_loader)

In [ ]:
@torch.no_grad()
def test_model(model, testloader):

    model.eval()
    criterion = nn.MSELoss()
    
    total_loss = 0.0
    voltage_predictions = []
    voltage_targets = []
    power_predictions = []
    power_targets = []
    
    for batch in testloader:

        batch = batch.to(device)

        # Forward pass
        pred_dict = model(batch)

        voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
        voltage_up = voltage_up.to(device)
        voltage_down = voltage_down.to(device)
        voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
        voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)

        power_up, power_down = convert_power_bounds(batch['generator'].x)
        power_up = power_up.to(device)
        power_down = power_down.to(device)
        powers = pred_dict['generator'] * (power_up - power_down) + power_down
        powers = torch.clamp(powers,min=power_down, max=power_up)
        
        combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
        combined_outputs = torch.cat([voltages, powers], dim=0)
        loss = criterion(combined_targets, combined_outputs)
        
        total_loss += loss.item()

        # Store predictions and targets for overall metrics
        voltage_predictions.append(voltages.cpu())
        voltage_targets.append(batch['bus'].y.cpu())

        power_predictions.append(powers.cpu())
        power_targets.append(batch['generator'].y.cpu())

    
    return total_loss / len(testloader), voltage_predictions, voltage_targets, power_predictions, power_targets

In [ ]:
model = HeteroInteractGNN().to(device)

In [ ]:
tot_params = 0
for parameter in model.parameters():
  layer_ws = 1
  for val in parameter.shape:
      layer_ws*=val
  tot_params += layer_ws
print(f"Total number of parameters = {tot_params}")

In [ ]:
# loaded_checkpoint = torch.load(f"/home/oarowolo/workfile/OPFData/checkpoints/checkpoint_epoch_40.pth")
# torch.save(loaded_checkpoint, f"{system_size}_bus_latest_N-1_checkpoint.pth")

In [ ]:
# model.load_state_dict(torch.load(f"/home/oarowolo/workfile/OPFData/{system_size}_bus_HeteroGNN_5_256_PQVT.pth"))

In [ ]:
# model.eval()

In [ ]:
# ## check our many unique list of generator indices there are in the test data and also how many unique list of edge configurations
# unique_gen_confs = set(tuple(sublist) for sublist in train_generator_indices)
# print('number of unique generator configurations is :',len(unique_gen_confs))

In [ ]:
# unique_lines_confs = set(tuple(sublist) for sublist in train_grid_ac_line_receivers)
# print('number of unique line configurations is :',len(unique_lines_confs))

In [ ]:
# unique_transformer_confs = set(tuple(sublist) for sublist in train_grid_transformer_receivers)
# print('number of unique transformer configurations is :',len(unique_transformer_confs))

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5,weight_decay=5e-8)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

In [ ]:
from pathlib import Path
def save_checkpoint(model, optimizer, scheduler, epoch, loss):
    """
    Save model checkpoint including all training state
    """
    # Create checkpoint directory if it doesn't exist
    # Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'loss': loss,
        'learning_rate': optimizer.param_groups[0]['lr']  # Current LR
    }
    
    # Save with epoch number in filename
    checkpoint_path = f'checkpoint_epoch_{epoch}.pth'
    torch.save(checkpoint, checkpoint_path)
    
    # Also save as 'latest' for easy loading
    latest_path = 'checkpoint_latest.pth'
    torch.save(checkpoint, latest_path)
    
    print(f"Checkpoint saved at epoch {epoch}: {checkpoint_path}")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    """
    Load checkpoint and restore training state
    """
    if not os.path.exists(checkpoint_path):
        print(f"No checkpoint found at {checkpoint_path}")
        return 0
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load optimizer state
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if it exists
    if scheduler and checkpoint['scheduler_state_dict']:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    lr = checkpoint['learning_rate']
    
    print(f"Loaded checkpoint from epoch {epoch}, loss: {loss:.4f}, lr: {lr:.6f}")
    
    return epoch + 1  # Return next epoch to start from

In [ ]:
training_losses = []
validation_losses = []
best_valid_loss = float('inf')
early_stop_thresh = 100
best_epoch = -1
best_model_state = None
num_epochs = 100

# new_epoch = load_checkpoint(model, optimizer, scheduler, f"/home/oarowolo/workfile/OPFData/checkpoints/checkpoint_epoch_40.pth")

for epoch in tqdm(range(num_epochs), desc="Training Progress"):
    train_loss = train_model(model, train_loader, optimizer)
    valid_loss = validate_model(model, val_loader)
    training_losses.append(train_loss)
    validation_losses.append(valid_loss)

    wandb.log({"training_loss": train_loss, "validation_loss": valid_loss})

    scheduler.step(train_loss)

    if epoch % 10 == 0:
      print(f'Epoch: {epoch}')
      print(f'\tTrain Loss: {train_loss:.4f}')
      print(f'\t Val. Loss: {valid_loss:.4f}')
    if valid_loss < best_valid_loss:
      best_valid_loss = valid_loss
      best_model_state = deepcopy(model.state_dict())
    
    # Save checkpoint every 10 epochs
    if epoch % 10 == 0:
        save_checkpoint(model, optimizer, scheduler, epoch, valid_loss)


plt.subplots(figsize=(5,3))
plt.plot([i for i in range(len(training_losses))], training_losses, 'r', label='Training loss')
plt.plot([i for i in range(len(validation_losses))], validation_losses, 'g', label='Validation loss')
plt.legend()
plt.title(f'GNN Training and Validation loss',fontsize = 15)
plt.xlabel('Epochs',fontsize = 12)
plt.ylabel('MSE Loss',fontsize = 12)
plt.semilogy()

training_losses=np.array(training_losses)
validation_losses=np.array(validation_losses)

model.load_state_dict(best_model_state)
model.eval()

In [ ]:
torch.save(model.state_dict(), f"{system_size}_bus_N-1_HybridHeteroGNN_5_256_PQVT.pth")
wandb.save(f"{system_size}_bus_N-1_HybridHeteroGNN_5_256_PQVT.pth")  # Upload to WandB

In [ ]:
test_loss, v_predictions, v_targets,p_predictions, p_targets = test_model(model, test_loader)

In [ ]:
print('loss on test data is ', test_loss)

In [ ]:
v_predictions = torch.cat(v_predictions, dim=0)
v_targets = torch.cat(v_targets, dim=0)

In [ ]:
p_predictions = torch.cat(p_predictions, dim=0)
p_targets = torch.cat(p_targets, dim=0)

In [ ]:
# compute voltage magnitude loss  and voltage angle loss separately
calc_loss = nn.MSELoss()
active_power_loss = calc_loss(p_predictions[:,0],p_targets[:,0])
reactive_power_loss = calc_loss(p_predictions[:,1],p_targets[:,1])

In [ ]:
print('average active power discrepancy is  ', active_power_loss)
print('average reactive power discrepancy is  ', reactive_power_loss)

In [ ]:
num_nodes = [n.shape[0] for n in test_generator_indices]

In [ ]:
#Iterate over num_nodes to extract each graph's predictions
separate_p_predictions = []
index = 0  # To track position in concatenated predictions

for nodes in num_nodes:
    separate_p_predictions.append(p_predictions[index:index + nodes])  # Extract corresponding predictions
    index += nodes  # Move index forward


In [ ]:
#Iterate over num_nodes to extract each graph's predictions
separate_p_targets = []
index = 0  # To track position in concatenated predictions

for nodes in num_nodes:
    separate_p_targets.append(p_targets[index:index + nodes])  # Extract corresponding predictions
    index += nodes  # Move index forward

In [ ]:
v_predictions = v_predictions.reshape(-1,system_size, 2)
v_targets = v_targets.reshape(-1, system_size, 2)

In [ ]:
# compute voltage magnitude loss  and voltage angle loss separately
calc_loss = nn.MSELoss()
voltage_angle_loss = calc_loss(v_predictions[:,:,0],v_targets[:,:,0])
voltage_magnitude_loss = calc_loss(v_predictions[:,:,1],v_targets[:,:,1])

In [ ]:
print('average voltage angle discrepancy is  ', voltage_angle_loss)
print('average voltage magnitude discrepancy is  ', voltage_magnitude_loss)

In [ ]:
def calculate_generator_power(
    demand: torch.Tensor, 
    voltage: torch.Tensor,  
    branches: list,  
    Yks: torch.Tensor,  
    Yij: torch.Tensor,  
    Yijc: torch.Tensor,  
    Tij: torch.Tensor,  
) -> torch.Tensor:
    num_nodes = voltage.shape[0]
    generator_power = torch.zeros_like(demand)
    
    # Helper function for complex multiplication
    def complex_mult(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return torch.stack([
            a[0] * b[0] - a[1] * b[1],
            a[0] * b[1] + a[1] * b[0]
        ])

    # Helper function for complex conjugate
    def complex_conj(x: torch.Tensor) -> torch.Tensor:
        return torch.stack([x[0], -x[1]])

    # Helper function for complex division
    def complex_div(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        denominator = b[0]**2 + b[1]**2
        return torch.stack([
            (a[0] * b[0] + a[1] * b[1]) / denominator,
            (a[1] * b[0] - a[0] * b[1]) / denominator
        ])
    
    # Calculate shunt power terms for each node
    shunt_power = torch.zeros_like(demand)
    for i in range(num_nodes):
        v_mag_sq = voltage[i, 0]**2 + voltage[i, 1]**2
        v_mag_sq_tensor = torch.tensor([v_mag_sq, 0.0])
        shunt_power[i] = complex_mult(complex_conj(Yks[i]), v_mag_sq_tensor)
    # print('the shunt powers are: ', shunt_power)
    
    # Calculate branch flows
    branch_flows = {} 
    
    for idx, (i, j) in enumerate(branches):
        # Get complex voltage at both ends
        vi = voltage[i]
        vj = voltage[j]
        
        # First term of Sij
        vi_mag_sq = vi[0]**2 + vi[1]**2
        vi_mag_sq_tensor = torch.tensor([vi_mag_sq, 0.0], dtype=torch.float64)
        tij_mag_sq = Tij[idx, 0]**2 + Tij[idx, 1]**2
        tij_mag_sq_tensor = torch.tensor([tij_mag_sq, 0.0], dtype=torch.float64)
        vi_over_tij_sq = complex_div(vi_mag_sq_tensor, tij_mag_sq_tensor)
        
        # Sum of branch admittance and charging admittance
        Y_total = torch.stack([
            Yij[idx, 0] + Yijc[idx, 0],
            Yij[idx, 1] + Yijc[idx, 1]
        ])
        term1 = complex_mult(complex_conj(Y_total), vi_over_tij_sq)
        
        # Second term of Sij
        vivj = complex_mult(vi, complex_conj(vj))
        term2 = complex_mult(
            complex_conj(Yij[idx]),
            complex_div(vivj, Tij[idx])
        )
        
        # Total branch flow Sij
        Sij = term1 - term2
        branch_flows[(i, j, idx)] = Sij
        # print(f'the branch flows for {i} , {j} in forward direction are: ', Sij)
        
        # Reverse flow Sji
        term1 = complex_mult(complex_conj(Y_total), vi_over_tij_sq)
        vj_mag_sq = vj[0]**2 + vj[1]**2
        vj_mag_sq_tensor = torch.tensor([vj_mag_sq, 0.0], dtype=torch.float64)
        
        term1_ji = complex_mult(complex_conj(Y_total), vj_mag_sq_tensor)
        vjvi = complex_mult(complex_conj(vi), vj)
        term2_ji = complex_mult(
            complex_conj(Yij[idx]),
            complex_div(vjvi, complex_conj(Tij[idx]))
        )
        Sji = term1_ji - term2_ji
        branch_flows[(j, i, idx)] = Sji
        # print(f'the branch flows for {j} , {i} in reverse direction are: ', Sji)

    # Aggregate generator power for each node
    for i in range(num_nodes):
        for index, (from_bus, to_bus) in enumerate(branches): 
            if from_bus == i:
                generator_power[i] += branch_flows[(from_bus, to_bus, index)]
            if to_bus == i:
                generator_power[i] += branch_flows[(to_bus, from_bus, index)]
        generator_power[i] += demand[i] + shunt_power[i]
    
    return generator_power, branch_flows


In [ ]:
def convert_to_complex_voltage(voltage_tensor):
    # Extract angle and magnitude
    voltage_angle = voltage_tensor[:,0:1]  # In radians
    voltage_magnitude = voltage_tensor[:,1:]
    
    # Calculate real and imaginary parts
    real_voltage = voltage_magnitude * torch.cos(voltage_angle)
    imaginary_voltage = voltage_magnitude * torch.sin(voltage_angle)
    
    return torch.concat((real_voltage,imaginary_voltage),dim=1)

In [ ]:
def convert_to_complex_rectangle(tensor_2d):
    # Extract angle and magnitude
    tensor_mag = tensor_2d[:,0:1]  # In radians
    tensor_angle = tensor_2d[:,1:]
    
    # Calculate real and imaginary parts
    real_tensor = tensor_mag * torch.cos(tensor_angle)
    imaginary_tensor = tensor_mag * torch.sin(tensor_angle)
    
    return torch.concat((real_tensor,imaginary_tensor),dim=1)

In [ ]:
def compute_gandb(edge_inputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
test_demand = torch.zeros((len(test_grid_load),system_size,test_grid_load[0].shape[-1]))

for k in range(len(test_grid_load)):
    instant_demand = test_demand[k]
    all_load_indices = test_load_indices[0].astype(int)
    
    instant_demand[all_load_indices] = torch.tensor(test_grid_load[k])
    

In [ ]:
test_shunt = torch.zeros((len(test_grid_shunt),system_size,test_grid_shunt[0].shape[-1]))

for k in range(len(test_grid_shunt)):
    instant_shunt = test_shunt[k]
    all_shunt_indices = test_shunt_indices[0].astype(int)
    
    instant_shunt[all_shunt_indices] = torch.tensor(test_grid_shunt[k])
    

In [ ]:
test_shunt = test_shunt[:,:, [1, 0]]

In [ ]:
def compute_homogenous_edges(branch_list,grid_ac_line_features,grid_transformer_features):

    edge_inputs = np.zeros((len(branch_list),11))
    
    edge_inputs[:grid_ac_line_features.shape[0],:9] = grid_ac_line_features  # rearranging edge inputs to align for transformers and transmission lines
    edge_inputs[grid_ac_line_features.shape[0]:,:2] =  grid_transformer_features[:,:2]
    edge_inputs[grid_ac_line_features.shape[0]:,2:4] =  grid_transformer_features[:,9:]
    edge_inputs[grid_ac_line_features.shape[0]:,4:9] =  grid_transformer_features[:,2:7]
    edge_inputs[grid_ac_line_features.shape[0]:,9:] =  grid_transformer_features[:,7:9]
    edge_inputs[:grid_ac_line_features.shape[0],9:10] = 1.0
    
    return torch.tensor(edge_inputs)

In [ ]:
test_branch_list = list(master_branch_list[i] for i in test_indices)

In [ ]:
load_input = test_demand.cpu()

In [ ]:
voltage_predictions = v_predictions.cpu()

In [ ]:
Gen_Powers = []

Branch_Flows = []

In [ ]:
for r in range(len(test_branch_list)):
    Branches = test_branch_list[r]
    present_edge_input = compute_homogenous_edges(Branches,test_grid_ac_line_features[r],test_grid_transformer_features[r])
    edge_g, edge_b = compute_gandb(present_edge_input)
    conductance_susceptance = torch.cat((edge_g, edge_b), dim=1)
    conductance_susceptance = conductance_susceptance.to('cpu')
    charging_susceptance = torch.zeros_like(conductance_susceptance)
    charging_susceptance[:,1:] =  present_edge_input[:,2:3].to('cpu')
    Tij = present_edge_input[:,9:].to('cpu')
    Tij_rec = convert_to_complex_rectangle(Tij)
    complex_v = convert_to_complex_voltage(v_predictions[r])
    Yks = test_shunt[r].to('cpu')
    load= load_input[r].to('cpu')
    gen_injection,branch_flows = calculate_generator_power(load,complex_v,Branches,Yks,conductance_susceptance,charging_susceptance,Tij_rec)
    Gen_Powers.append(gen_injection)
    Branch_Flows.append(branch_flows)

In [ ]:
generator_power_balance = torch.stack(Gen_Powers,dim=0)

In [ ]:
max_generators_per_node = 0
for k in range(len(separate_p_targets)):
    unique_indices, counts = torch.unique(torch.tensor(test_generator_indices[k]), return_counts=True)
    max_generators_per_node = max(max_generators_per_node, counts.max().item())
# Create tensor with additional dimension to hold multiple generators per node
test_generator_cost = torch.zeros(len(test_grid_generator), system_size, max_generators_per_node, 3)

# Fill the tensor with generator cost values
for k in range(len(test_grid_generator)):
    instant_gen_cost = test_generator_cost[k]
    this_gen_cost = test_grid_generator[k]
    
    instant_gen_indices = test_generator_indices[k]
    
    # Track how many generators we've already seen at each node
    node_counts = torch.zeros(system_size, dtype=torch.long)
    
    # Assign each generator cost to its proper location
    for i, idx in enumerate(instant_gen_indices):
        # Place this generator's cost in the next available slot for this node
        generator_slot = node_counts[idx]
        instant_gen_cost[idx, generator_slot] = torch.tensor(this_gen_cost[i, 8:])
        # Increment the count for this node
        node_counts[idx] += 1


In [ ]:
# First, determine the maximum number of generators at any single location
max_generators_per_node = 0
for k in range(len(separate_p_targets)):
    unique_indices, counts = torch.unique(torch.tensor(test_generator_indices[k]), return_counts=True)
    max_generators_per_node = max(max_generators_per_node, counts.max().item())

# Create tensor with additional dimension to hold multiple generators per node
test_powers = torch.zeros((len(separate_p_predictions), system_size, max_generators_per_node, separate_p_predictions[0].shape[-1]))

# Fill the tensor with generator values
for k in range(len(separate_p_predictions)):
    instant_power = test_powers[k]
    instant_gen_indices = test_generator_indices[k]
    
    # Track how many generators we've already seen at each node
    node_counts = torch.zeros(system_size, dtype=torch.long)
    
    # Assign each generator output to its proper location
    for i, idx in enumerate(instant_gen_indices):
        # Place this generator's output in the next available slot for this node
        generator_slot = node_counts[idx]
        instant_power[idx, generator_slot] = separate_p_predictions[k][i]
        # Increment the count for this node
        node_counts[idx] += 1

In [ ]:
# First, determine the maximum number of generators at any single location
max_generators_per_node = 0
for k in range(len(separate_p_targets)):
    unique_indices, counts = torch.unique(torch.tensor(test_generator_indices[k]), return_counts=True)
    max_generators_per_node = max(max_generators_per_node, counts.max().item())

# Create tensor with additional dimension to hold multiple generators per node
target_powers = torch.zeros((len(separate_p_targets), system_size, max_generators_per_node, separate_p_targets[0].shape[-1]))

# Fill the tensor with generator values
for k in range(len(separate_p_targets)):
    instant_power = target_powers[k]
    instant_gen_indices = test_generator_indices[k]
    
    # Track how many generators we've already seen at each node
    node_counts = torch.zeros(system_size, dtype=torch.long)
    
    # Assign each generator output to its proper location
    for i, idx in enumerate(instant_gen_indices):
        # Place this generator's output in the next available slot for this node
        generator_slot = node_counts[idx]
        instant_power[idx, generator_slot] = separate_p_targets[k][i]
        # Increment the count for this node
        node_counts[idx] += 1

In [ ]:
def compute_optimality(test_inputs, predictions, test_objective):

    test_inputs = test_inputs.cpu()
    test_objective = test_objective.cpu()

    c2 = test_inputs[:,:,:,0:1]
    c1 = test_inputs[:,:,:,1:2]
    c0 = test_inputs[:,:,:,2:3]

    # Get relevant output dimensions (zero-indexed)
    p_gens = predictions[:,:,:,0:1] # select on Pgs for generators
  

    print('number of nonzero  c2 is ',  torch.count_nonzero(c2[0]))
    print('the shape of p_gen is ', p_gens.shape)
    
    # Compute node-wise metrics
    system_metrics = c2 * (p_gens ** 2) + c1 * p_gens + c0
    print('the shape of system_metrics is ', system_metrics.shape)


    model_obj = torch.sum(system_metrics, dim=(1,2)).view(-1, 1)

    print('the shape of test objective is ', test_objective.shape)
    print('the shape of model objective is ', model_obj.shape)

    print(f'average model objective is {model_obj.mean()}')
    print(f'average IPOPT objective is {test_objective.mean()}')

    optimality_gap = (model_obj / test_objective) * 100

    
    return optimality_gap.mean()

In [ ]:
test_objective = torch.tensor(solution_objective[test_indices].astype(np.float32))

In [ ]:
opt_gap = compute_optimality(test_generator_cost, test_powers, test_objective)

In [ ]:
print('optimality gap now is ', opt_gap)

In [ ]:
predicted_angle_differences = []
predicted_angles = v_predictions[:,:,0]

for j in range(v_predictions.shape[0]):
    
    branch_for_this = test_branch_list[j]
    angle_differences = calculate_angle_differences(predicted_angles[j],branch_for_this)
    angle_differences = torch.tensor(angle_differences)
    predicted_angle_differences.append(angle_differences)
    

In [ ]:
true_angle_differences = []
true_angles = v_targets[:,:,0]

for j in range(v_targets.shape[0]):
    
    branch_for_this = test_branch_list[j]
    angle_differences = calculate_angle_differences(true_angles[j],branch_for_this)
    angle_differences = torch.tensor(angle_differences)
    true_angle_differences.append(angle_differences)

In [ ]:
predicted_angle_differences = torch.cat(predicted_angle_differences)
true_angle_differences = torch.cat(true_angle_differences)

In [ ]:
predicted_angle_differences.max()

In [ ]:
# voltage angle difference bound 


angle_diff_upper = torch.full(predicted_angle_differences.shape, 0.5236)
angle_diff_lower = torch.full(predicted_angle_differences.shape, -0.5236)

# Calculate violations
lower_angle_violations = torch.clamp(angle_diff_lower - predicted_angle_differences, min=0)  # Positive if below lower bound
upper_angle_violations = torch.clamp(predicted_angle_differences - angle_diff_upper, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
angle_diff_violations = lower_angle_violations + upper_angle_violations

print('max voltage angle difference violation is : ',angle_diff_violations.max() )
print('average voltage angle difference violation is : ',angle_diff_violations.mean())

In [ ]:
test_buses = torch.tensor(np.array(test_grid_bus))

In [ ]:
## voltage magnitude bound

vmin = test_buses[:,:,2:3].to('cpu')

vmax = test_buses[:,:,3:4].to('cpu')

lower_vmag_violation = torch.clamp(vmin - v_predictions[:,:,1:2], min=0)  # Positive if below lower bound

upper_vmag_violation = torch.clamp(v_predictions[:,:,1:2] - vmax, min=0)  # Positive if above upper bound

vmag_violations = lower_vmag_violation + upper_vmag_violation

print('max voltage magnitude violation is : ',vmag_violations.max() )
print('average voltage magnitude violation is : ',vmag_violations.mean() )

In [ ]:
max_generators_per_node = 0
for k in range(len(test_grid_generator)):
    unique_indices, counts = torch.unique(torch.tensor(test_generator_indices[k]), return_counts=True)
    max_generators_per_node = max(max_generators_per_node, counts.max().item())
# Create tensor with additional dimension to hold multiple generators per node
test_generator_inputs = torch.zeros(len(test_grid_generator), system_size, max_generators_per_node, 11)

# Fill the tensor with generator cost values
for k in range(len(test_grid_generator)):
    instant_gen_input = test_generator_inputs[k]
    this_gen_input = test_grid_generator[k]
    
    instant_gen_indices = test_generator_indices[k]
    
    # Track how many generators we've already seen at each node
    node_counts = torch.zeros(system_size, dtype=torch.long)
    
    # Assign each generator cost to its proper location
    for i, idx in enumerate(instant_gen_indices):
        # Place this generator's cost in the next available slot for this node
        generator_slot = node_counts[idx]
        instant_gen_input[idx, generator_slot] = torch.tensor(this_gen_input[i, :])
        # Increment the count for this node
        node_counts[idx] += 1


In [ ]:
# Gen active power bounds 
pmin = test_generator_inputs[:,:,:,2:3].to('cpu')

pmax = test_generator_inputs[:,:,:,3:4].to('cpu')


p_gens = test_powers[:,:,:,0:1]


lower_pgen_violations = torch.clamp(pmin - p_gens, min=0)  # Positive if below lower bound
upper_pgen_violations = torch.clamp(p_gens - pmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
pgen_violations = lower_pgen_violations + upper_pgen_violations

print('max gen active power violation is : ',pgen_violations.max())
print('average active power violation is : ',pgen_violations.mean())

In [ ]:
# Gen reactive power bounds 
qmin = test_generator_inputs[:,:,:,5:6].to('cpu')

qmax = test_generator_inputs[:,:,:,6:7].to('cpu')


q_gens = test_powers[:,:,:,1:2]


lower_qgen_violations = torch.clamp(qmin - q_gens, min=0)  # Positive if below lower bound
upper_qgen_violations = torch.clamp(q_gens - qmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
qgen_violations = lower_qgen_violations + upper_qgen_violations

print('max gen reactive power violation is : ',qgen_violations.max())
print('average reactive power violation is : ',qgen_violations.mean())

In [ ]:
# compute the power magnitudes

# Separate real and imaginary parts
def convert_to_power_magnitude(power_flow):
    
    real = power_flow[..., 0]  
    imag = power_flow[..., 1]  

    # Compute the magnitudes
    magnitudes = torch.sqrt(real**2 + imag**2) 


    result_tensor = magnitudes.unsqueeze(-1)

    return result_tensor

In [ ]:
for_flow_vio = []
rev_flow_vio = []

In [ ]:
long_term_line_rating = []

for k in range(len(test_grid_ac_line_features)):

    all_lines = test_grid_ac_line_features[k]
    line_thermal_ratings = torch.tensor(all_lines[:,6:7])
    all_transformers = test_grid_transformer_features[k]
    trans_thermal_ratings = torch.tensor(all_transformers[:,4:5])

    all_edge_ratings = torch.cat((line_thermal_ratings, trans_thermal_ratings), dim=0)
    long_term_line_rating.append(all_edge_ratings)



In [ ]:
for k in range(len(Branch_Flows)):
    flow_branch = test_branch_list[k]
    forward_keys = [(i, j, index) for index, (i,j) in enumerate(flow_branch)]
    reverse_keys = [(j, i, index) for index, (i,j) in enumerate(flow_branch)]
    forward_branch_flows = {key: Branch_Flows[k][key] for key in forward_keys if key in Branch_Flows[k]}
    reverse_branch_flows = {key: Branch_Flows[k][key] for key in reverse_keys if key in Branch_Flows[k]}


    forward_power_flows_list = [tensor for tensor in forward_branch_flows.values()]
    
    
    # Step 2: Concatenate tensors along the second axis (dim=1)
    forward_power_flows = torch.cat(forward_power_flows_list, dim=0).reshape(-1,2)



    reverse_power_flows = [tensor for tensor in reverse_branch_flows.values()]
    # Step 2: Concatenate tensors along the second axis (dim=1)
    reverse_power_flows = torch.cat(reverse_power_flows, dim=0).reshape(-1,2)
    forward_flow_magnitude = convert_to_power_magnitude(forward_power_flows)
    reverse_flow_magnitude = convert_to_power_magnitude(reverse_power_flows)
    # Branch flow bounds in forward direction


    branch_flow_limit = long_term_line_rating[k]
    forward_branch_flow = forward_flow_magnitude
    forward_flow_violations = torch.clamp(forward_branch_flow - branch_flow_limit, min=0)  # Positive if above upper bound
    for_flow_vio.append(forward_flow_violations)
    reverse_branch_flow = reverse_flow_magnitude
    reverse_flow_violations = torch.clamp(reverse_flow_magnitude - branch_flow_limit, min=0)  # Positive if above upper bound
    rev_flow_vio.append(reverse_flow_violations)



In [ ]:
for_flow_vio = torch.cat(for_flow_vio)
rev_flow_vio = torch.cat(rev_flow_vio)

In [ ]:
print('max forward power flow violation is : ',for_flow_vio.max() )
print('average  forward power flow violation is : ',for_flow_vio.mean() )

In [ ]:
print('max reverse power flow violation is : ', rev_flow_vio.max() )
print('average reverse power flow violation is : ', rev_flow_vio.mean() )

In [ ]:
test_power_per_node = torch.sum(test_powers, dim=2)

In [ ]:
true_power_per_node = torch.sum(target_powers, dim=2)

In [ ]:
## Evaluate power balance contraint violations
real_power_balance_mismatches = generator_power_balance[:,:,0] - test_power_per_node[:,:,0]


print('max active power balance mismatch is : ', real_power_balance_mismatches.max() )
print('average active power balance mismatch is : ', real_power_balance_mismatches.mean())

In [ ]:
reactive_power_balance_mismatches = generator_power_balance[:,:,1] - test_power_per_node[:,:,1]


print('max reactive power balance mismatch is : ', reactive_power_balance_mismatches.max() )
print('average reactive power balance mismatch is : ', reactive_power_balance_mismatches.mean())

In [ ]:
#create table to save important metrics
columns=["metric", "value"]
model_metrics_table = wandb.Table(columns=columns)

In [ ]:
model_metrics_table.add_data("optimality gap", opt_gap)
model_metrics_table.add_data("max voltage angle difference violation", angle_diff_violations.max())
model_metrics_table.add_data("average voltage angle difference violation", angle_diff_violations.mean())
model_metrics_table.add_data("max voltage magnitude violation", vmag_violations.max())
model_metrics_table.add_data("average voltage magnitude violation", vmag_violations.mean())
model_metrics_table.add_data("max gen active power violation", pgen_violations.max())
model_metrics_table.add_data("average gen active power violation", pgen_violations.mean())
model_metrics_table.add_data("max gen reactive power violation", qgen_violations.max())
model_metrics_table.add_data("average gen reactive power violation", qgen_violations.mean())
model_metrics_table.add_data("max forward power flows violation", for_flow_vio.max())
model_metrics_table.add_data("average forward power flows violation", for_flow_vio.mean())
model_metrics_table.add_data("max reverse power flows violation", rev_flow_vio.max())
model_metrics_table.add_data("average reverse power flows violation", rev_flow_vio.mean())
model_metrics_table.add_data("max active power balance mismatch", real_power_balance_mismatches.max())
model_metrics_table.add_data("average active power balance mismatch", real_power_balance_mismatches.mean())
model_metrics_table.add_data("max reactive power balance mismatch", reactive_power_balance_mismatches.max())
model_metrics_table.add_data("average reactive power balance mismatch", reactive_power_balance_mismatches.mean())
wandb.log({"model_metrics_table" : model_metrics_table})

In [ ]:
wandb.finish()

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print('hurray, done!')